# Train and Validate
* Train a model across the specified training sites and sampling approach
* Validate the model at the excluded site

## Todo
* Pull out probabilities using model.predict_proba(observations_to_predict)
* 

In [ ]:
import pathlib
import numpy
import dask.distributed
import pandas
import joblib

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Values to edit

In [ ]:
sample_method = "sampling_2"
method_2_threshold = .97 # 1.0 .99 .98 .97 .95

In [ ]:
# View the amount of training data for the selected sampling apporach
samples_per_site = pandas.read_csv(utils.get_samples_summary_file_path(sample_method, method_2_threshold))
samples_per_site

In [ ]:
samples_per_site.sum(axis=0)

In [ ]:
uav_classes_to_ignore = ['Shadow', 'Glare']

In [ ]:
satellite_classes_100 = {'Seagrass': 1, 'Gracilaria': 2, 'Ulva': 3, 'Satmarsh': 4, 'Unvegetated': 5, 'Terrestrial': 6, 'Water': 7, 'Mixed': 8}
satellite_classes_from_uav_classes_100 = {
    'Seagrass': ['Seagrass'],
    'Gracilaria': ['Gracilaria'],
    'Ulva': ['Ulva', 'Ulva mats'],
    'Satmarsh': ['Saltmarsh', ],
    'Terrestrial': ['Terrestrial'],
    'Unvegetated': ['Unvegetated'],                            
    'Water': ['Water', 'Submerged vegetation', 'Gracilaria submerged', 'Seagrass submerged'],
    'Mixed': ['Cystophora', 'Hormosira','Brown algae mixed', 'Microphytobenthos', 'Green algae mixed', 
          'Filamentous brown algae', 'Red algae mixed', 'Rock'],
}

In [ ]:
all_training_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

In [ ]:
training_sites_akaroa = ["Duvauchelle", "Robinsons", "Childrens", "Takamatua"] 
training_sites_SI = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai"] 
all_sites_excluding_ihuta =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

### Make sure you change the `samples_name` and `satellite_class_name` names when experimenting

In [ ]:
satellite_classes=satellite_classes_100
satellite_from_uav_classes=satellite_classes_from_uav_classes_100

In [ ]:
sample_sites = all_sites_excluding_ihuta # ["Ihutai"] all_training_sites all_sites_excluding_ihuta

samples_name = "all_sites_excluding_ihuta" # "ihutai" "all_training_sites" "all_sites_excluding_ihuta"
satellite_class_name = "test"

# Cells to run
* Combine training data and code `satellite_class_id`
* Plot
  * satellite bands of each UAV class
  * satellite bands of each satellite class

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()
uav_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
sample_folder = utils.get_samples_path(sample_method=sample_method, method_2_threshold=method_2_threshold)

spectral_plot_folder = utils.get_spectral_plots_path(sample_method=sample_method, method_2_threshold=method_2_threshold)

### Train and save model

In [ ]:
uav_plot_filename = spectral_plot_folder / f"{samples_name}.png"
satellite_plot_filename = spectral_plot_folder / f"{satellite_class_name}.png"

if not uav_plot_filename.exists() or not satellite_plot_filename.exists():
    # train and save the model
    samples_dataframe = training.map_satellite_ids_into_samples(
        training_sites=sample_sites,
        samples_path=sample_folder,
        uav_labels_file=uav_labels_file,
        uav_classes_to_ignore=uav_classes_to_ignore,
        satellite_classes=satellite_classes,
        satellite_from_uav_classes=satellite_from_uav_classes)
    
    # Save a record of the classes considered in training
    spectral_plot_folder.mkdir(exist_ok=True)
    satellite_classes_dataframe = pandas.DataFrame.from_dict(satellite_classes, orient='index', columns=['satellite_class_id'])
    satellite_classes_dataframe['uav_class_ids'] = satellite_classes_dataframe.index.map(lambda key: f"{satellite_from_uav_classes[key]}")
    satellite_classes_dataframe.to_csv(satellite_plot_filename.with_name(f"{satellite_plot_filename.stem}_class_mappings.csv"), index=True)

    uav_training_labels = (
            pandas.read_csv(uav_labels_file, sep="\t", header=None, names=["Value", "Key"])
            .set_index("Key")["Value"]
            .to_dict()
        )
    uav_classes_in_training = [next((key for key, value in uav_training_labels.items() if value == int(uav_id)), None) 
                               for uav_id in samples_dataframe["uav_class_id"].unique()]
    satellite_classes_in_training = [next((key for key, value in satellite_classes.items() if value == int(satellite_id)), None) 
                               for satellite_id in samples_dataframe["satellite_class_id"].unique()]
    
    print(f"\nSatellite classes present: {satellite_classes_in_training}")
    print(f"UAV classes present: {uav_classes_in_training}")

    if not uav_plot_filename.exists():
        # Save a summary of the UAV class totals in the samples
        samples_summary=pandas.DataFrame(samples_dataframe['uav_class_id'].value_counts())
        samples_summary["uav_class_name"] = samples_summary.index.map(lambda index: next((key for key, value in uav_training_labels.items() if value == int(index)), None) )
        samples_summary = samples_summary[["uav_class_name", "count"]] # Change column order
        samples_summary.to_csv(uav_plot_filename.with_suffix(f".csv"))
        print(samples_summary)
        training.save_samples_uav_classes(plot_filename=uav_plot_filename, training_dataframe=samples_dataframe, uav_labels_file=uav_labels_file)
    else:
        print(f"\nPlot {uav_plot_filename.name} already exist. Delete if you want to recreate it.")

    if not satellite_plot_filename.exists():
        training.save_samples_satellite_classes(plot_filename=satellite_plot_filename, training_dataframe=samples_dataframe, satellite_labels=satellite_classes)
    else:
        print(f"\nPlot {satellite_plot_filename.name} already exist. Delete if you want to recreate it.")

else:
    print(f"Plots {uav_plot_filename.name} and {satellite_plot_filename.name} already exist. Delete if you want to recreate it.")
    

